# Use the orchestration service of Generative AI Hub

The orchestration service of Generative AI Hub lets you use all the available models with the same codebase. You only deploy the orchestration service and then you can access all available models simply by changing the model name parameter. You can also use grounding, prompt templating, data masking and content filtering capabilities.

Store the `orchestration deployment url` from the previous step in your `variables.py` file.

The v2 API used in this notebook introduces a modular configuration design — LLM, template, and optional modules like grounding are composed independently and assembled into a `ModuleConfig`, making it easier to add, remove, or swap capabilities without rewriting the entire configuration.

This code is based on the [AI180 TechEd 2024 Jump-Start session](https://github.com/SAP-samples/teched2024-AI180/tree/e648921c46337b57f61ecc9a93251d4b838d7ad0/exercises/python).

In [ ]:
import init_env
import variables

init_env.set_environment_variables()

### Import the packages you want to use

In [ ]:
from gen_ai_hub.orchestration_v2.models.llm_model_details import LLMModelDetails
from gen_ai_hub.orchestration_v2.models.config import ModuleConfig, OrchestrationConfig
from gen_ai_hub.orchestration_v2.models.document_grounding import (
    DocumentGroundingFilter,
    DocumentGroundingConfig,
    DocumentGroundingPlaceholders,
    GroundingModuleConfig,
)
from gen_ai_hub.orchestration_v2.models.template import Template, PromptTemplatingModuleConfig
from gen_ai_hub.orchestration_v2.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration_v2.service import OrchestrationService

### Assign a model and define a prompt template
**user_query** is again the user input. Whereas **grounding_response** is the context retrieved from the context information, in this case from sap.help.com.

In [ ]:
# TODO again you need to chose a model e.g. gemini-2.5-flash or gpt-4o-mini
llm = LLMModelDetails(
    name="",
    params={
        'temperature': 0.0,
    }
)
template = Template(
    template=[
        SystemMessage(content="You are a helpful translation assistant."),
        UserMessage(content="""Answer the request by providing relevant answers that fit to the request.
                Request: {{?user_query}}
                Context:{{?grounding_response}}
                """),
    ]
)

### Create an orchestration configuration that specifies the grounding capability

In [ ]:
# Set up Document Grounding
filters = [
    DocumentGroundingFilter(id="SAPHelp", data_repository_type="help.sap.com")
]

grounding_config = GroundingModuleConfig(
    config=DocumentGroundingConfig(
        filters=filters,
        placeholders=DocumentGroundingPlaceholders(
            input=["user_query"],
            output="grounding_response",
        ),
    )
)

prompt_template = PromptTemplatingModuleConfig(prompt=template, model=llm)
config = OrchestrationConfig(
    modules=ModuleConfig(
        prompt_templating=prompt_template,
        grounding=grounding_config,
    )
)

In [ ]:
import importlib
variables = importlib.reload(variables)

orchestration_service = OrchestrationService(
    api_url=variables.AICORE_ORCHESTRATION_DEPLOYMENT_URL,
    config=config
)

response = orchestration_service.run(
    placeholder_values={
        #TODO Here you can change the user prompt into whatever you want to ask the model
        "user_query": "What is Joule?"
    }
)

print(response.final_result.choices[0].message.content)

# Streaming
Long response times can be frustrating in chat applications, especially when a large amount of text is involved. To create a smoother, more engaging user experience, we can use streaming to display the response in real-time, character by character or chunk by chunk, as the model generates it. This avoids the awkward wait for a complete response and provides immediate feedback to the user.

In [ ]:
from gen_ai_hub.orchestration_v2.models.streaming import GlobalStreamOptions

stream_config = OrchestrationConfig(
    modules=ModuleConfig(
        prompt_templating=prompt_template,
        grounding=grounding_config,
    ),
    stream=GlobalStreamOptions(enabled=True),
)

response = orchestration_service.stream(
    config=stream_config,
    placeholder_values={
        #TODO Here you can change the user prompt into whatever you want to ask the model
        "user_query": "What is Joule?"
    }
)

for chunk in response:
    if chunk.final_result.choices[0].delta.content:
        print(chunk.final_result.choices[0].delta.content, end="", flush=True)

[More Info on the content filter](https://learn.microsoft.com/en-us/azure/ai-studio/concepts/content-filtering)

[Next exercise](10-chatbot-with-memory.ipynb)